# Empirical Demonstration: Sequence Characteristics (pairfam family data)

This notebook shows **how to use the algorithms in `sequenzo.sequence_characteristics`** on real data. We use the **pairfam family-by-month** dataset: life-course family-formation sequences (264 months, 9 states: partnership and parenthood). Each section demonstrates one or more functions with brief explanations and interpreted results.

### Load data and set up sequence structure

Load the pairfam family-by-month dataset and inspect it. Then we define the time axis (264 months), the 9 states and their labels, and build a `SequenceData` object used by all `sequence_characteristics` functions.

In [9]:
# Imports: sequenzo provides load_dataset, SequenceData, and sequence_characteristics
from sequenzo import *
import pandas as pd

print("Available datasets in Sequenzo:", list_datasets())
df = load_dataset("pairfam_family_by_month")
df

Available datasets in Sequenzo: ['biofam', 'biofam_child_domain', 'biofam_left_domain', 'biofam_married_domain', 'chinese_colonial_territories', 'country_child_mortality_global_deciles', 'country_co2_emissions', 'country_co2_emissions_global_deciles', 'country_co2_emissions_global_quintiles', 'country_co2_emissions_local_deciles', 'country_co2_emissions_local_quintiles', 'country_gdp_per_capita_quintiles', 'country_life_expectancy_global_deciles', 'dyadic_children', 'dyadic_parents', 'mvad', 'pairfam_activity_by_month', 'pairfam_activity_by_year', 'pairfam_family_by_month', 'pairfam_family_by_year', 'political_science_aid_shock', 'political_science_donor_fragmentation', 'students_stress_states_by_week']


,id,weight40,sex,doby_gen,dob,ethni,migstatus,yeduc,sat1i4,sat5,...,255,256,257,258,259,260,261,262,263,264
0,111000.0,0.343964,1,1971,855,1,1,11.5,5,7,...,4,4,4,4,4,4,4,4,4,4
1,2931000.0,1.767455,0,1973,881,5,3,10.5,5,5,...,9,9,9,9,9,9,9,9,9,9
2,3491000.0,0.726561,1,1971,857,1,1,18.0,10,-2,...,2,2,2,2,2,2,2,2,2,2
3,3902000.0,0.940405,0,1971,853,1,1,16.0,8,5,...,9,9,9,9,9,9,9,9,9,9
4,4814000.0,1.373152,0,1973,886,1,1,11.5,8,5,...,9,9,9,9,9,9,9,9,9,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1022,918595000.0,0.505112,1,1972,864,1,1,11.5,7,10,...,1,1,1,1,1,1,1,1,1,1
1023,918958000.0,3.204625,0,1971,854,-7,-7,10.5,10,3,...,3,3,3,3,3,3,3,3,3,3
1024,919111000.0,1.472961,0,1972,875,1,1,11.5,5,3,...,6,6,6,6,6,6,6,6,6,6
1025,920140000.0,2.022823,0,1971,859,1,1,11.5,10,10,...,7,7,7,7,7,7,7,7,7,7


In [10]:
# --- Build SequenceData for use with sequence_characteristics ---
# Time span: 264 months (columns 1 ... 264)
time_list = [f"{i}" for i in range(1, 265)]

# Nine states (numeric codes 1–9) for partnership × parenthood
states = list(range(1, 10))

# Define labels for each state
labels = [
    "Single, no child",
    "Living apart together, no child",
    "Cohabiting, no child",
    "Married, no child",
    "Single, with child(ren)",
    "LAT, with child(ren)",
    "Cohabiting, with child(ren)",
    "Married, 1 child",
    "Married, 2+ children"
]

# Custom colors for the 9 states (optional; used in sequence plots)
colors = [
    "#74C9B4",  # mint green
    "#A6E3D0",  # light aqua
    "#F9E79F",  # apricot yellow
    "#F6CDA3",  # peach
    "#F5B7B1",  # dusty pink
    "#D7BDE2",  # light purple
    "#A3C4F3",  # sky blue
    "#7FB3D5",  # steel blue
    "#EAECEE"   # cloud white
]



# Initialize SequenceData (required input for all sequence_characteristics functions)
sequence_data = SequenceData(
    df,
    time=time_list,
    id_col="id",
    states=states,
    labels=labels,
    weights=df["weight40"].values,
    custom_colors=colors,
)



[>] SequenceData initialized successfully! Here's a summary:
[>] Number of sequences: 1027
[>] Number of time points: 264
[>] Min/Max sequence length: 264 / 264
[>] States: [1, 2, 3, 4, 5, 6, 7, 8, 9]
[>] Labels: ['Single, no child', 'Living apart together, no child', 'Cohabiting, no child', 'Married, no child', 'Single, with child(ren)', 'LAT, with child(ren)', 'Cohabiting, with child(ren)', 'Married, 1 child', 'Married, 2+ children']
[>] Weights: Provided (total weight=1244.245, mean=1.212, std=0.893)


## 1. Complexity index

**What it is:** A normalized measure of how “complex” each sequence is, combining the number of distinct subsequences and the number of transitions. Higher values mean more varied, less repetitive patterns.

**How to use:** Pass your `SequenceData` object. Returns a dataframe with one row per sequence: `ID` and `Complexity Index`.

In [11]:
get_complexity_index(sequence_data)

,ID,Complexity Index
0,111000.0,0.128952
1,2931000.0,0.142920
2,3491000.0,0.137828
3,3902000.0,0.157175
4,4814000.0,0.090977
...,...,...
1022,918595000.0,0.102699
1023,918958000.0,0.053411
1024,919111000.0,0.180456
1025,920140000.0,0.161840


## 2. Turbulence

**What it is:** Measures how “turbulent” each sequence is (weighted sum of state changes and entropy). Higher values indicate more instability or unpredictability over time.

**How to use:** Pass `SequenceData`. Returns a dataframe with `ID` and `Turbulence`.

In [12]:
get_turbulence(sequence_data)

[!] One or more missing values were found after calculating the number of distinct subsequences. They have been replaced with a large number of 1e15 to ensure the calculation continues.


,ID,Turbulence
0,111000.0,13.201720
1,2931000.0,11.855482
2,3491000.0,12.751613
3,3902000.0,13.651798
4,4814000.0,6.969324
...,...,...
1022,918595000.0,8.487718
1023,918958000.0,4.855986
1024,919111000.0,16.194609
1025,920140000.0,17.672635


## 3. State frequencies and within-sequence entropy

**What it is:** For each sequence, the time (or count) spent in each state and a **within-sequence entropy** summarizing how evenly time is spread across states. One row per sequence; columns are state labels (counts or durations) plus an entropy column.

**How to use:** Pass `SequenceData`. Useful for describing individual life courses and comparing diversity across sequences.

In [13]:
get_state_freq_and_entropy_per_seq(sequence_data)

[>] Computing state distribution for 1027 sequences and 9 states ...


,ID,"Single, no child","Living apart together, no child","Cohabiting, no child","Married, no child","Single, with child(ren)","LAT, with child(ren)","Cohabiting, with child(ren)","Married, 1 child","Married, 2+ children"
0,111000.0,0.0,128.0,0.0,118.0,10.0,3.0,0.0,5.0,0.0
1,2931000.0,123.0,0.0,36.0,0.0,14.0,0.0,9.0,45.0,37.0
2,3491000.0,67.0,43.0,115.0,0.0,29.0,10.0,0.0,0.0,0.0
3,3902000.0,65.0,0.0,97.0,0.0,14.0,0.0,2.0,16.0,70.0
4,4814000.0,16.0,0.0,50.0,0.0,0.0,0.0,12.0,32.0,154.0
...,...,...,...,...,...,...,...,...,...,...
1022,918595000.0,47.0,0.0,103.0,0.0,17.0,0.0,97.0,0.0,0.0
1023,918958000.0,201.0,0.0,63.0,0.0,0.0,0.0,0.0,0.0,0.0
1024,919111000.0,63.0,26.0,9.0,25.0,18.0,123.0,0.0,0.0,0.0
1025,920140000.0,50.0,0.0,67.0,0.0,133.0,0.0,14.0,0.0,0.0


## 4. Cross-sectional entropy

**What it is:** At each time point, how the **population** is distributed across states (not within one sequence). Entropy at time \(t\) is high when many states are present in similar proportions; it is low when one state dominates. The function prints a short summary and returns a long-format dataframe: `time`, `state`, `freq`, `per_time_entropy_norm`, etc.

**How to use:** Pass `SequenceData`. Use the returned table to plot entropy over time or to see which state is dominant at each \(t\).

In [14]:
result = get_cross_sectional_entropy(sequence_data)
result


Cross-Sectional Entropy Summary
[>] Number of states: 9
[>] Number of time points: 264
[>] On average, the most common state accounts for 32.6% of cases
[>] Entropy is highest at time point 159
[>] Entropy is lowest at time point 1
[>] Average normalized entropy: 0.787 (range: 0 = fully concentrated, 1 = evenly distributed)



,time,state,freq,per_time_entropy_norm,N_valid,Effective States,rank,is_top
0,1,1,0.633503,0.373916,1244.245137,2.274083,1,True
1,1,3,0.322965,0.373916,1244.245137,2.274083,2,False
2,1,5,0.035802,0.373916,1244.245137,2.274083,3,False
3,1,4,0.003182,0.373916,1244.245137,2.274083,4,False
4,1,7,0.002455,0.373916,1244.245137,2.274083,5,False
...,...,...,...,...,...,...,...,...
2371,99,8,0.096875,0.840714,1244.245137,6.342268,5,False
2372,99,9,0.047144,0.840714,1244.245137,6.342268,6,False
2373,99,6,0.040024,0.840714,1244.245137,6.342268,7,False
2374,99,2,0.011668,0.840714,1244.245137,6.342268,8,False


### Summary (from cross-sectional entropy)

Interpretation of the cross-sectional entropy run above: **9 states**, **264 time points**; on average the dominant state accounts for **~33%** of the (weighted) population at a given time; **entropy is highest at t=159** and **lowest at t=1** (early in the life course the distribution is more concentrated).

In [15]:
# Optional: print the same summary (re-run cross-sectional entropy first to refresh result)
# print("Summary: 9 states, 264 time points; dominant state ~33%; entropy max at t=159, min at t=1.")